# 🌿 Crop Disease Detection — YOLO26 Training Notebook

> **Dataset:** Ghana Crop Disease Challenge v2 (Roboflow) — 4,901 images  
> **Classes:** 23 disease/health categories across Corn, Pepper, and Tomato  
> **Model:** Ultralytics YOLO26 (latest architecture, January 2026)  
> **Task:** Object Detection with Out-of-Distribution (OOD) rejection

---

## ⚠️ Problem Being Solved

Standard YOLO is a **closed-set classifier** — it will *always* try to assign an input region to one of your trained classes. This means that if you show it a human face or an unrelated object, it will still attempt to force-fit it to the nearest crop disease class.

This notebook addresses that with a multi-pronged strategy:
1. **Hard Negative Mining** — collecting images with no crop content and training the model to ignore them
2. **High confidence threshold** — raising the inference bar so low-confidence guesses are discarded
3. **Optimised hyperparameters** — getting the best possible accuracy on the known classes so far fewer false positives remain
4. **Post-inference OOD filter** — an optional secondary check that compares max confidence against an adaptive threshold

---

## 📦 Install Dependencies

In [ ]:
# Install / upgrade Ultralytics (includes YOLO26 support)
# !pip install -q --upgrade ultralytics

# Additional utilities used later in this notebook
# !pip install -q matplotlib seaborn pandas pillow tqdm scipy

import ultralytics
ultralytics.checks()   # prints YOLO version, GPU info, etc.

## 📁 Paths & Configuration

In [ ]:
import os
from pathlib import Path

# ─── Project Root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()          # assumes notebook is run from project root
DATA_DIR     = PROJECT_ROOT / "data" / "main"
DATA_YAML    = DATA_DIR / "data.yaml"
NEG_DIR      = PROJECT_ROOT / "data" / "negatives"  # hard-negative images (no labels)

# ─── Training output ───────────────────────────────────────────────────────────
RUNS_DIR     = PROJECT_ROOT / "runs"
EXP_NAME     = "crop_disease_yolo26"

# ─── Model ─────────────────────────────────────────────────────────────────────
# YOLO26 model sizes: yolo26n, yolo26s, yolo26m, yolo26l, yolo26x
# For best accuracy while staying trainable on a single GPU, yolo26m is a great default.
# Switch to yolo26l / yolo26x if you have an A100 / H100.
MODEL_SIZE   = "yolo26n"   # n=fastest; switch to yolo26m/l/x if val mAP plateaus (underfitting)

# ─── Training hypers ───────────────────────────────────────────────────────────
EPOCHS       = 150         # Enough for the dataset; early-stop saves time
IMG_SIZE     = 640         # Matches the Roboflow pre-processing (resize 640x640)
BATCH        = 16          # Reduce to 8 if you hit GPU OOM
PATIENCE     = 30          # Early-stop patience (epochs without mAP improvement)

# ─── Inference OOD rejection threshold ─────────────────────────────────────────
# Any detection with confidence below this is discarded at inference time.
# Raising this aggressively reduces false positives on unrelated images.
CONF_THRESHOLD = 0.50      # 50 % — tune upward if you still see false positives
IOU_THRESHOLD  = 0.45      # NMS IoU (YOLO26 is NMS-free but kept for compat)

# ─── Class labels (from data.yaml) ─────────────────────────────────────────────
CLASS_NAMES = [
    'Corn_Cercospora_Leaf_Spot', 'Corn_Common_Rust', 'Corn_Healthy',
    'Corn_Northern_Leaf_Blight', 'Corn_Streak',
    'Pepper_Bacterial_Spot', 'Pepper_Cercospora', 'Pepper_Early_Blight',
    'Pepper_Fusarium', 'Pepper_Healthy', 'Pepper_Late_Blight',
    'Pepper_Leaf_Blight', 'Pepper_Leaf_Curl', 'Pepper_Leaf_Mosaic',
    'Pepper_Septoria',
    'Tomato_Bacterial_Spot', 'Tomato_Early_Blight', 'Tomato_Fusarium',
    'Tomato_Healthy', 'Tomato_Late_Blight', 'Tomato_Leaf_Curl',
    'Tomato_Mosaic', 'Tomato_Septoria'
]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data YAML    : {DATA_YAML}")
print(f"Model        : {MODEL_SIZE}")
print(f"Classes      : {NUM_CLASSES}")
assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}"
# ─── Device (auto-detect multi-GPU / single GPU / MPS / CPU) ──────────────────
import torch

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        # Multi-GPU: train across all available CUDA GPUs
        DEVICE = list(range(n_gpus))    # e.g. [0, 1, 2, 3]
        BATCH  = 16 * n_gpus            # 16 samples per GPU
        print(f"Multi-GPU        : {n_gpus} GPUs → device={DEVICE}")
    else:
        DEVICE = 0                      # single NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"                      # Apple Silicon GPU
    n_gpus = 0
else:
    DEVICE = "cpu"
    n_gpus = 0

print(f"Device           : {DEVICE}")
print(f"Batch size       : {BATCH}")


## 🗂️ Verify & Inspect Dataset

In [ ]:
import yaml
import collections
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

# ─── Load YAML ─────────────────────────────────────────────────────────────────
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print("data.yaml contents:\n", yaml.dump(cfg, default_flow_style=False))

# ─── Count annotations per split ───────────────────────────────────────────────
def count_annotations(split):
    label_dir = DATA_DIR / split / "labels"
    img_dir   = DATA_DIR / split / "images"
    counts = collections.Counter()
    total_imgs = 0
    empty_imgs = 0
    for lf in label_dir.glob("*.txt"):
        total_imgs += 1
        lines = lf.read_text().strip().split("\n")
        if lines == ['']:
            empty_imgs += 1
            continue
        for line in lines:
            if line.strip():
                cls_id = int(line.split()[0])
                counts[CLASS_NAMES[cls_id]] += 1
    return counts, total_imgs, empty_imgs

for split in ["train", "valid", "test"]:
    counts, total, empty = count_annotations(split)
    print(f"\n── {split.upper()} SPLIT ──")
    print(f"  Total images : {total}")
    print(f"  Empty labels : {empty}")
    print(f"  Total boxes  : {sum(counts.values())}")
    for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"    {cls:<45} {n:>5}")

# ─── Class distribution bar chart ──────────────────────────────────────────────
train_counts, _, _ = count_annotations("train")
labels = list(train_counts.keys())
values = [train_counts[l] for l in labels]

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#2ecc71' if 'Healthy' in l else '#e74c3c' if 'Corn' in l else '#3498db' if 'Pepper' in l else '#9b59b6'
          for l in labels]
bars = ax.barh(labels, values, color=colors)
ax.set_xlabel('Number of Bounding Boxes', fontsize=12)
ax.set_title('Training Set — Class Distribution', fontsize=14, fontweight='bold')
ax.invert_yaxis()
patches = [
    mpatches.Patch(color='#e74c3c', label='Corn'),
    mpatches.Patch(color='#3498db', label='Pepper'),
    mpatches.Patch(color='#9b59b6', label='Tomato'),
    mpatches.Patch(color='#2ecc71', label='Healthy'),
]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "class_distribution.png", dpi=150)
plt.show()
print("Saved class_distribution.png")

## 🚫 Prepare Hard-Negative Images (OOD Rejection)

This is the **most important step** to fix the false-positive problem.

We collect images that are *not* crops (faces, animals, buildings, random objects, etc.) and add them to the training set **without any bounding-box labels**. The model then learns that these patterns correspond to "nothing to detect" instead of forcing them into a crop-disease category.

Options:
- **Option A (recommended):** Download negatives from the OpenImages or COCO dataset automatically (code below)
- **Option B:** Manually drop your own images into `data/negatives/images/` — they must have **no corresponding label files**

In [ ]:
import urllib.request
import random
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─── Create negatives directory structure ──────────────────────────────────────
NEG_IMG_DIR = NEG_DIR / "images"
NEG_LBL_DIR = NEG_DIR / "labels"   # intentionally kept empty
NEG_IMG_DIR.mkdir(parents=True, exist_ok=True)
NEG_LBL_DIR.mkdir(parents=True, exist_ok=True)

# ─── Option A: Download from picsum.photos (random diverse images) ─────────────
# picsum delivers random nature/urban/people images — perfect negatives.
# Uses a fixed seed so the same 300 targets are chosen every run,
# allowing safe resumption without re-downloading already-saved files.

NUM_NEGATIVES = 300   # how many negative images to add

# Fixed seed → same 300 seeds every run → resumption works correctly
random.seed(42)
seeds = random.sample(range(1, 2000), NUM_NEGATIVES)

# Identify which images still need downloading
pending = [(seed, NEG_IMG_DIR / f"negative_{seed:04d}.jpg")
           for seed in seeds
           if not (NEG_IMG_DIR / f"negative_{seed:04d}.jpg").exists()]

already = NUM_NEGATIVES - len(pending)
print(f"Already downloaded : {already}/{NUM_NEGATIVES}")
print(f"Remaining to fetch  : {len(pending)}")

TIMEOUT = 20   # seconds per request
MAX_WORKERS = 8  # parallel download threads

def _download(args):
    seed, dest = args
    url = f"https://picsum.photos/seed/{seed}/640/640"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
            data = resp.read()
        dest.write_bytes(data)
        return seed, None
    except Exception as e:
        return seed, str(e)

if pending:
    ok = err = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_download, item): item for item in pending}
        for done_count, future in enumerate(as_completed(futures), 1):
            seed, exc = future.result()
            if exc:
                print(f"  ⚠️  seed {seed}: {exc}")
                err += 1
            else:
                ok += 1
            if done_count % 50 == 0 or done_count == len(pending):
                print(f"  {done_count}/{len(pending)} processed  "
                      f"(ok={ok}, errors={err})")
    print(f"\nDownload complete: {ok} new images, {err} errors")

existing = list(NEG_IMG_DIR.glob("*.jpg"))
print(f"\nNegatives directory : {NEG_IMG_DIR}")
print(f"Total images ready  : {len(existing)}")

# ─── Copy negatives into the TRAIN split ───────────────────────────────────────
# Empty label files will NOT be created — YOLO interprets missing labels as
# 'background-only' images and uses them for negative sampling.
TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
TRAIN_LBL_DIR = DATA_DIR / "train" / "labels"

copied = 0
for img_path in existing:
    dst = TRAIN_IMG_DIR / img_path.name
    if not dst.exists():
        shutil.copy2(img_path, dst)
        # Create an EMPTY label file — tells YOLO this image has no objects
        (TRAIN_LBL_DIR / img_path.with_suffix(".txt").name).touch()
        copied += 1

print(f"Copied {copied} new negatives into {TRAIN_IMG_DIR}")
print("✅  Hard-negative setup complete.")

## 🛠️ Fix data.yaml Paths

In [ ]:
import yaml

# Rewrite data.yaml with absolute paths so it works regardless of CWD
fixed_yaml = {
    "path"  : str(DATA_DIR),                         # dataset root
    "train" : str(DATA_DIR / "train" / "images"),
    "val"   : str(DATA_DIR / "valid" / "images"),
    "test"  : str(DATA_DIR / "test"  / "images"),
    "nc"    : NUM_CLASSES,
    "names" : CLASS_NAMES,
}

FIXED_YAML = PROJECT_ROOT / "data_fixed.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.dump(fixed_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Fixed YAML written to: {FIXED_YAML}")
print(yaml.dump(fixed_yaml, default_flow_style=False, sort_keys=False))

## 🔍 Visualise Sample Training Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
TRAIN_LBL_DIR = DATA_DIR / "train" / "labels"

img_files = sorted(TRAIN_IMG_DIR.glob("*.jpg"))[:500]
sample = random.sample(img_files, min(9, len(img_files)))

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
colors = plt.cm.get_cmap('tab20', NUM_CLASSES)

for ax, img_path in zip(axes.flatten(), sample):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    ax.imshow(img)
    
    lbl_path = TRAIN_LBL_DIR / img_path.with_suffix(".txt").name
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split("\n"):
            if not line.strip():
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = (cx - bw/2) * W
            y1 = (cy - bh/2) * H
            rect = patches.Rectangle(
                (x1, y1), bw*W, bh*H,
                linewidth=2, edgecolor=colors(cls_id), facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, max(y1-4, 0), CLASS_NAMES[cls_id].split('_')[-1],
                    fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.1', fc=colors(cls_id), alpha=0.8))
    ax.axis('off')
    ax.set_title(img_path.name[:30], fontsize=8)

plt.suptitle('Sample Training Images with Annotations', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "sample_annotations.png", dpi=120)
plt.show()

## 🚀 Train YOLO26

### Key hyperparameters explained

| Parameter | Value | Why |
|-----------|-------|-----|
| `model` | `yolo26m` | Best accuracy/speed trade-off for most GPUs |
| `epochs` | 150 | Enough for 4.9k images; early-stop exits early |
| `patience` | 30 | Stop if mAP50-95 doesn't improve for 30 epochs |
| `imgsz` | 640 | Matches dataset resolution |
| `batch` | 16 | Reduce to 8 if you hit OOM |
| `lr0` | 0.01 | Starting LR — YOLO26's MuSGD handles it well |
| `lrf` | 0.01 | Final LR ratio (cosine decay) |
| `warmup_epochs` | 5 | Gradual LR warmup for stable early training |
| `hsv_h/s/v` | varied | Colour jitter helps generalise across lighting |
| `degrees` | 10 | Mild rotation — leaves in various orientations |
| `mosaic` | 1.0 | Mosaic augmentation — very effective for crops |
| `mixup` | 0.15 | MixUp blending — improves boundary decisions |
| `copy_paste` | 0.3 | Copy-paste augmentation — helps small lesion detection |
| `label_smoothing` | 0.1 | Prevents overconfident predictions → better OOD behaviour |
| `cls` | 0.3 | Lower class loss weight — focuses on localisation |
| `weight_decay` | 0.001 | L2 regularisation → reduces overfitting |

In [ ]:
import os, time
from ultralytics import YOLO

# ─── Dry-run mode ──────────────────────────────────────────────────────────────
# Set DRY_RUN=1 (env var) or use train.py --dry-run to validate the pipeline and
# time one epoch before committing to a full 200-epoch run.
_DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"
_EPOCHS  = 1 if _DRY_RUN else EPOCHS
if _DRY_RUN:
    print("⚡  DRY-RUN mode — running 1 epoch to validate pipeline")

# ─── Resume or fresh start ─────────────────────────────────────────────────────
# If training was interrupted, last.pt holds the most recent checkpoint.
# resume=True restores optimizer state, epoch counter, and all hyperparameters
# from the saved args.yaml — no re-specification needed.
LAST_PT = RUNS_DIR / EXP_NAME / "weights" / "last.pt"
RESUME  = LAST_PT.exists()

if RESUME:
    print(f"Resuming interrupted training from:\n  {LAST_PT}")
    model = YOLO(str(LAST_PT))
    t0    = time.perf_counter()
    results = model.train(resume=True)

else:
    model = YOLO(f"{MODEL_SIZE}.pt")   # downloads pretrained weights automatically

    _is_mps   = DEVICE == "mps"
    _is_multi = isinstance(DEVICE, list)

    # ── Startup log ────────────────────────────────────────────────────────────
    _n_train = len(list((DATA_DIR / "train" / "images").glob("*.*")))
    _n_val   = len(list((DATA_DIR / "valid" / "images").glob("*.*")))
    print(f"\n{\'─\'*60}")
    print(f"  Device       : {DEVICE}")
    print(f"  Model        : {MODEL_SIZE}.pt")
    print(f"  Batch        : {BATCH}{\'  (16/GPU)' if _is_multi else ''}")
    print(f"  Image size   : {IMG_SIZE}×{IMG_SIZE}")
    print(f"  Train images : {_n_train}")
    print(f"  Val images   : {_n_val}")
    print(f"  Epochs       : {_EPOCHS}{\'  ← DRY RUN' if _DRY_RUN else ''}")
    print(f"  Cache        : ram")
    print(f"  AMP          : True")
    print(f"{\'─\'*60}\n")

    # ────────────────────────────────────────────────────────────────────────────
    # Non-default hyperparameter choices — one-line rationale for each
    # ────────────────────────────────────────────────────────────────────────────
    # device=mps/[0,1,…] : Apple Silicon GPU (MPS) or all available CUDA GPUs
    # batch=16*n_gpus    : 16 images per GPU — fits M4 Pro 24 GB comfortably
    # workers=0          : MPS requires 0 — macOS fork() conflicts with Metal
    #                      (workers=8 is used automatically when DEVICE is CUDA)
    # cache="ram"        : ~4 GB of images loaded once into RAM; fastest I/O
    #                      swap to cache="disk" if Activity Monitor shows pressure
    # amp=True           : fp16 autocast → ~1.4× speedup; set False if NaN losses
    # epochs=200         : cosine LR needs room to decay with a 3k-image dataset
    # patience=25        : early-stop after 25 non-improving epochs (was 30)
    # freeze=10          : lock first 10 backbone layers; faster, less overfit risk
    #                      set to 0 if val mAP plateaus and train mAP is fine
    # close_mosaic=10    : disable mosaic in last 10 epochs for fine-grain tuning
    # label_smoothing=0.1: prevents overconfident predictions — key for OOD guard
    # seed=42            : reproducible runs for fair hyperparameter comparisons
    # deterministic      : False under DDP (multi-GPU); True otherwise
    # ────────────────────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    results = model.train(
        # ── Data ──
        data        = str(FIXED_YAML),
        imgsz       = IMG_SIZE,
        batch       = BATCH,
        cache       = "ram",              # load to RAM; swap to "disk" if pressure
        workers     = 0 if _is_mps else 8,

        # ── Training schedule ──
        epochs      = _EPOCHS,
        patience    = 25,                 # stop after 25 non-improving epochs
        optimizer   = "MuSGD",           # YOLO26 hybrid Muon+SGD optimiser
        lr0         = 0.01,
        lrf         = 0.01,
        momentum    = 0.937,
        weight_decay= 0.001,
        warmup_epochs   = 5,
        warmup_momentum = 0.8,
        cos_lr      = True,

        # ── Regularisation / fine-tuning ──
        freeze      = 10,                 # freeze first 10 backbone layers
        close_mosaic= 10,                 # disable mosaic in final 10 epochs

        # ── Augmentation ──
        hsv_h       = 0.015,
        hsv_s       = 0.7,
        hsv_v       = 0.4,
        degrees     = 10.0,
        translate   = 0.1,
        scale       = 0.5,
        shear       = 0.0,
        perspective = 0.0001,
        flipud      = 0.0,
        fliplr      = 0.5,
        bgr         = 0.0,
        mosaic      = 1.0,
        mixup       = 0.15,
        copy_paste  = 0.3,
        erasing     = 0.4,

        # ── Loss weights ──
        box         = 7.5,
        cls         = 0.3,
        dfl         = 1.5,

        # ── OOD / confidence ──
        label_smoothing = 0.1,
        conf        = None,

        # ── Misc ──
        device      = DEVICE,
        project     = str(RUNS_DIR),
        name        = EXP_NAME,
        exist_ok    = True,
        pretrained  = True,
        amp         = True,               # fp16; set False if NaN losses appear
        verbose     = True,
        plots       = True,
        save        = True,
        save_period = 10,
        val         = True,
        seed        = 42,
        deterministic = not _is_multi,
    )

elapsed = time.perf_counter() - t0

if _DRY_RUN:
    print(f"\n{\'─\'*60}")
    print(f"  Dry-run epoch time   : {elapsed:.1f}s")
    print(f"  Estimated full run   : ~{elapsed * EPOCHS / 60:.0f} min  ({elapsed * EPOCHS / 3600:.1f} h)")
    print(f"  Best weights         : {results.save_dir}/weights/best.pt")
    print(f"{\'─\'*60}\n")
else:
    print(f"\n✅  Training complete!  ({elapsed/3600:.1f} h total)")
    print(f"   Best model: {results.save_dir}/weights/best.pt")


## 📊 Training Results & Plots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

save_dir = Path(results.save_dir)
csv_path = save_dir / "results.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print(df.tail(10).to_string(index=False))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics = [
        ('train/box_loss',  'Box Loss (train)',  '#e74c3c'),
        ('train/cls_loss',  'Class Loss (train)', '#3498db'),
        ('metrics/mAP50',   'mAP@0.50',          '#2ecc71'),
        ('metrics/mAP50-95','mAP@0.50:0.95',     '#9b59b6'),
        ('metrics/precision','Precision',         '#f39c12'),
        ('metrics/recall',  'Recall',             '#1abc9c'),
    ]
    for ax, (col, title, color) in zip(axes.flatten(), metrics):
        if col in df.columns:
            ax.plot(df[col], color=color, linewidth=2)
            ax.set_title(title, fontsize=13, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
    plt.suptitle(f'Training Metrics — {EXP_NAME}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_dir / "training_summary.png", dpi=150)
    plt.show()

# ─── Show confusion matrix if present ─────────────────────────────────────────
cm_path = save_dir / "confusion_matrix_normalized.png"
if cm_path.exists():
    img = Image.open(cm_path)
    plt.figure(figsize=(14, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Normalised Confusion Matrix', fontsize=14)
    plt.show()

## ✅ Validate on Val & Test Sets

In [ ]:
from ultralytics import YOLO

# Load the best checkpoint
best_model_path = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

print("\n─── Validation set ───")
val_results = best_model.val(
    data    = str(FIXED_YAML),
    split   = "val",
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    conf    = CONF_THRESHOLD,
    iou     = IOU_THRESHOLD,
    device  = 0,
    plots   = True,
)
print(f"Val mAP50   : {val_results.box.map50:.4f}")
print(f"Val mAP50-95: {val_results.box.map:.4f}")

print("\n─── Test set ───")
test_results = best_model.val(
    data    = str(FIXED_YAML),
    split   = "test",
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    conf    = CONF_THRESHOLD,
    iou     = IOU_THRESHOLD,
    device  = 0,
    plots   = True,
)
print(f"Test mAP50   : {test_results.box.map50:.4f}")
print(f"Test mAP50-95: {test_results.box.map:.4f}")

# Per-class AP
print("\n─── Per-class AP50 ───")
for cls, ap in zip(CLASS_NAMES, test_results.box.ap50):
    bar = '█' * int(ap * 30)
    print(f"  {cls:<45} {ap:.3f}  {bar}")

## 🧪 OOD Rejection Test

This cell tests the model with images that are **NOT crops** to verify the false-positive fix.  
A detection is only accepted if its confidence ≥ `CONF_THRESHOLD` (0.50 by default).  
Tune this value higher if you still see false positives.

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import tempfile, io

# ─── OOD images to test (clearly not crops) ────────────────────────────────────
OOD_URLS = [
    "https://picsum.photos/seed/101/640/640",
    "https://picsum.photos/seed/202/640/640",
    "https://picsum.photos/seed/303/640/640",
    "https://picsum.photos/seed/404/640/640",
    "https://picsum.photos/seed/505/640/640",
    "https://picsum.photos/seed/606/640/640",
]

def ood_inference(model, img_pil, conf_thr=0.50):
    """Run inference and apply the OOD confidence gate."""
    results = model.predict(
        source = np.array(img_pil),
        conf   = conf_thr,
        iou    = IOU_THRESHOLD,
        verbose= False,
        device = DEVICE,
    )
    return results[0]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, url in zip(axes.flatten(), OOD_URLS):
    try:
        with urllib.request.urlopen(url) as resp:
            img = Image.open(io.BytesIO(resp.read())).convert("RGB")
    except Exception as e:
        ax.text(0.5, 0.5, f"Load error:\n{e}", ha='center', va='center')
        ax.axis('off')
        continue

    result = ood_inference(best_model, img, conf_thr=CONF_THRESHOLD)
    n_det  = len(result.boxes) if result.boxes is not None else 0

    ax.imshow(img)
    if n_det > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls  = int(box.cls[0])
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"{CLASS_NAMES[cls]} {conf:.2f}",
                    fontsize=8, color='red',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
        ax.set_title(f"⚠️  {n_det} detections (conf≥{CONF_THRESHOLD})",
                     color='red', fontsize=10, fontweight='bold')
    else:
        ax.set_title(f"✅  No detections (conf≥{CONF_THRESHOLD})",
                     color='green', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'OOD Rejection Test  —  conf_threshold = {CONF_THRESHOLD}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "ood_test_results.png", dpi=120)
plt.show()

## 🔬 Confidence Calibration

Scans the validation set to find the confidence threshold that maximises F1 on **known crop images** while minimising detections on **negative images**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

VALID_IMG_DIR = DATA_DIR / "valid" / "images"
VALID_LBL_DIR = DATA_DIR / "valid" / "labels"

val_imgs = list(VALID_IMG_DIR.glob("*.jpg"))[:200]

all_confs_positive = []
all_confs_negative = []

for img_path in val_imgs:
    lbl_path = VALID_LBL_DIR / img_path.with_suffix(".txt").name
    has_objects = lbl_path.exists() and lbl_path.read_text().strip() != ''

    res = best_model.predict(
        source=str(img_path), conf=0.01, iou=IOU_THRESHOLD,
        verbose=False, device=DEVICE
    )[0]

    confs = res.boxes.conf.tolist() if res.boxes and len(res.boxes) > 0 else []

    if has_objects:
        all_confs_positive.extend(confs)
    else:
        all_confs_negative.extend(confs)

fig, ax = plt.subplots(figsize=(12, 5))
bins = np.linspace(0, 1, 51)
if all_confs_positive:
    ax.hist(all_confs_positive, bins=bins, alpha=0.6, color='#2ecc71', label='True crop detections')
if all_confs_negative:
    ax.hist(all_confs_negative, bins=bins, alpha=0.6, color='#e74c3c', label='False positives (empty imgs)')
ax.axvline(x=CONF_THRESHOLD, color='navy', linestyle='--', linewidth=2,
           label=f'Current threshold ({CONF_THRESHOLD})')
ax.set_xlabel('Confidence Score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Confidence Score Distribution — Positive vs Negative Images', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "confidence_calibration.png", dpi=150)
plt.show()

if all_confs_positive:
    p5  = np.percentile(all_confs_positive, 5)
    p50 = np.percentile(all_confs_positive, 50)
    print(f"Positive detections  — 5th pct: {p5:.3f}, median: {p50:.3f}")
if all_confs_negative:
    p95 = np.percentile(all_confs_negative, 95)
    print(f"Negative detections  — 95th pct: {p95:.3f}")
    print(f"\n💡 Recommendation: set CONF_THRESHOLD ≥ max({p95:.2f}+0.05, {CONF_THRESHOLD})")

## 🖼️ Inference Demo

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import random

TEST_IMG_DIR = DATA_DIR / "test" / "images"
test_imgs = list(TEST_IMG_DIR.glob("*.jpg"))
demo_imgs = random.sample(test_imgs, min(6, len(test_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, img_path in zip(axes.flatten(), demo_imgs):
    img = Image.open(img_path).convert("RGB")
    result = best_model.predict(
        source  = str(img_path),
        conf    = CONF_THRESHOLD,
        iou     = IOU_THRESHOLD,
        verbose = False,
        device  = DEVICE,
    )[0]

    ax.imshow(img)
    if result.boxes and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls  = int(box.cls[0])
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2.5, edgecolor='#f39c12', facecolor='none')
            ax.add_patch(rect)
            label_text = f"{CLASS_NAMES[cls].replace('_', ' ')}\n{conf:.2f}"
            ax.text(x1+2, y1+2, label_text, fontsize=7.5, color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.3', fc='#2c3e50', alpha=0.85))

    ax.set_title(img_path.stem[:35], fontsize=9)
    ax.axis('off')

plt.suptitle(f'Inference Results  |  conf ≥ {CONF_THRESHOLD}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "inference_demo.png", dpi=120)
plt.show()

## 📋 Production Inference Helper

Ready-to-use inference function that wraps the model with the OOD rejection guard.  
Copy this into your application code.

In [ ]:
from ultralytics import YOLO
import numpy as np
from PIL import Image
from pathlib import Path

CLASS_NAMES_PROD = [
    'Corn_Cercospora_Leaf_Spot', 'Corn_Common_Rust', 'Corn_Healthy',
    'Corn_Northern_Leaf_Blight', 'Corn_Streak',
    'Pepper_Bacterial_Spot', 'Pepper_Cercospora', 'Pepper_Early_Blight',
    'Pepper_Fusarium', 'Pepper_Healthy', 'Pepper_Late_Blight',
    'Pepper_Leaf_Blight', 'Pepper_Leaf_Curl', 'Pepper_Leaf_Mosaic',
    'Pepper_Septoria',
    'Tomato_Bacterial_Spot', 'Tomato_Early_Blight', 'Tomato_Fusarium',
    'Tomato_Healthy', 'Tomato_Late_Blight', 'Tomato_Leaf_Curl',
    'Tomato_Mosaic', 'Tomato_Septoria'
]


class CropDiseaseDetector:
    """
    Thin wrapper around YOLO26 with OOD rejection.

    Parameters
    ----------
    model_path     : path to best.pt
    conf_threshold : detections below this are discarded (OOD guard)
    iou_threshold  : NMS IoU threshold
    device         : 'cpu', 0, 'cuda:0', etc.
    """

    def __init__(
        self,
        model_path: str,
        conf_threshold: float = 0.50,
        iou_threshold:  float = 0.45,
        device = DEVICE,
    ):
        self.model     = YOLO(model_path)
        self.conf_thr  = conf_threshold
        self.iou_thr   = iou_threshold
        self.device    = device

    def predict(self, image_source) -> list[dict]:
        results = self.model.predict(
            source  = image_source,
            conf    = self.conf_thr,
            iou     = self.iou_thr,
            device  = self.device,
            verbose = False,
        )
        detections = []
        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return []

        for box in result.boxes:
            cls_id  = int(box.cls[0])
            conf    = float(box.conf[0])
            xyxy    = box.xyxy[0].tolist()
            detections.append({
                "class_id"   : cls_id,
                "class_name" : CLASS_NAMES_PROD[cls_id],
                "confidence" : round(conf, 4),
                "bbox_xyxy"  : [round(v, 2) for v in xyxy],
            })
        return detections

    def is_crop_image(self, image_source, min_confidence: float = 0.40) -> bool:
        """Returns True if the image appears to contain a crop leaf."""
        dets = self.predict(image_source)
        if not dets:
            return False
        return max(d["confidence"] for d in dets) >= min_confidence


# ─── Example usage ─────────────────────────────────────────────────────────────
best_pt = str(Path(results.save_dir) / "weights" / "best.pt")
detector = CropDiseaseDetector(model_path=best_pt, conf_threshold=CONF_THRESHOLD)

test_sample = random.choice(list((DATA_DIR / "test" / "images").glob("*.jpg")))
print("\n─── Crop image test ───")
dets = detector.predict(str(test_sample))
print(f"Image : {test_sample.name}")
if dets:
    for d in dets:
        print(f"  → {d['class_name']} | conf={d['confidence']:.3f} | box={d['bbox_xyxy']}")
else:
    print("  No detections (image below confidence threshold)")
print(f"\nis_crop_image: {detector.is_crop_image(str(test_sample))}")

## 📈 Summary & Next Steps

### What was done to fix the OOD (false-positive) problem

| Strategy | Where | Effect |
|----------|-------|--------|
| **Hard Negative Mining** | Prepare negatives section | Model sees 300 diverse non-crop images with empty labels → learns to predict nothing on them |
| **Label Smoothing (0.1)** | Train YOLO26 | Forces the model to be less overconfident; OOD images get lower max-conf |
| **High confidence threshold (0.50)** | Inference | Any prediction below 50% is discarded — catches most false positives |
| **Copy-paste + MixUp augmentation** | Train YOLO26 | Better boundary learning → fewer spurious detections |
| **`is_crop_image()` helper** | Production inference | Application-level pre-filter before running full detection |

### Tuning guide

- **Still getting false positives?**
  - Raise `CONF_THRESHOLD` to `0.60` or `0.70`
  - Add more negative images (raise `NUM_NEGATIVES` to 600+)
  - Re-train with `label_smoothing=0.15`

- **Missing real diseases (false negatives)?**  
  - Lower `CONF_THRESHOLD` back toward `0.35`
  - Use a larger model (`yolo26l` or `yolo26x`)
  - Train more epochs or add more annotated disease images

---
> **Next step:** Run `export.ipynb` to convert the trained model to ExecuTorch `.pte` format for deployment in your Android/Kotlin app.

## 📊 Publication Figures — Dataset & Training Analysis

Generates and saves **nine publication-quality figures** to `yolo_output/`.  
Run this section any time (before or after training); figures that reference  
`results` (training metrics) are guarded with an existence check.

| Figure | File | Content |
|--------|------|---------|
| Fig 01 | `fig_01_dataset_overview.png` | Split sizes, hard-negatives, box counts |
| Fig 02 | `fig_02_class_distribution_train.png` | Per-class box count (training set) |
| Fig 03 | `fig_03_cross_split_distribution.png` | Absolute + normalised cross-split comparison |
| Fig 04 | `fig_04_annotation_density.png` | Boxes-per-image histogram (train & val) |
| Fig 05 | `fig_05_bbox_spatial_heatmap.png` | Spatial distribution of box centres |
| Fig 06 | `fig_06_bbox_size_analysis.png` | Box width/height scatter, area & aspect ratio |
| Fig 07 | `fig_07_class_imbalance.png` | Per-class imbalance ratio across splits |
| Fig 08 | `fig_08_training_config.png` | Training hyperparameter summary table |
| Fig 09 | `fig_09_lr_schedule_augmentation.png` | LR curve + augmentation profile |

In [ ]:
# ── Publication Figures: Setup ────────────────────────────────────────────────
import collections
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

# Output directory
OUTPUT_DIR = PROJECT_ROOT / "yolo_output"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Publication rcParams (300 DPI save, clean axes) ───────────────────────────
plt.rcParams.update({
    "font.family"        : "DejaVu Sans",
    "font.size"          : 11,
    "axes.titlesize"     : 13,
    "axes.titleweight"   : "bold",
    "axes.labelsize"     : 11,
    "xtick.labelsize"    : 10,
    "ytick.labelsize"    : 10,
    "legend.fontsize"    : 10,
    "legend.framealpha"  : 0.9,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "savefig.pad_inches" : 0.15,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "axes.grid"          : True,
    "grid.alpha"         : 0.3,
    "grid.linestyle"     : "--",
    "axes.axisbelow"     : True,
})

# ── Colour palette ────────────────────────────────────────────────────────────
CROP_PALETTE  = {"Corn": "#E8973A", "Pepper": "#27AE60", "Tomato": "#C0392B"}
SPLIT_PALETTE = {"train": "#2980B9", "valid": "#27AE60", "test":  "#E74C3C"}
HEALTHY_HEX   = "#3498DB"

def _class_color(name):
    if "Healthy" in name:
        return HEALTHY_HEX
    for crop, col in CROP_PALETTE.items():
        if name.startswith(crop):
            return col
    return "#95A5A6"

CLASS_COLORS = [_class_color(c) for c in CLASS_NAMES]

def _get_crop(name):
    return name.split("_")[0]

# ── Load all YOLO annotations into a DataFrame ────────────────────────────────
def _load_split(split):
    ldir = DATA_DIR / split / "labels"
    rows = []
    for lf in sorted(ldir.glob("*.txt")):
        lines = [ln.strip() for ln in lf.read_text().split("\n") if ln.strip()]
        if not lines:
            rows.append(dict(split=split, image=lf.stem, class_id=-1,
                             class_name="background", crop="none",
                             cx=np.nan, cy=np.nan,
                             bw=np.nan, bh=np.nan, area=np.nan))
            continue
        for line in lines:
            p = line.split()
            cid = int(p[0])
            cx, cy, bw, bh = map(float, p[1:5])
            cname = CLASS_NAMES[cid]
            rows.append(dict(split=split, image=lf.stem,
                             class_id=cid, class_name=cname, crop=_get_crop(cname),
                             cx=cx, cy=cy, bw=bw, bh=bh, area=bw * bh))
    return pd.DataFrame(rows)

print("Loading annotation data …")
_dfs     = {s: _load_split(s) for s in ["train", "valid", "test"]}
_df_all  = pd.concat(_dfs.values(), ignore_index=True)
_df_box  = _df_all[_df_all.class_id >= 0].copy()   # exclude background rows
_n_imgs  = {s: len(list((DATA_DIR / s / "images").glob("*.*")))
            for s in ["train", "valid", "test"]}

for s, df in _dfs.items():
    n_b = (df.class_id >= 0).sum()
    n_e = (df.class_id <  0).sum()
    print(f"  {s:6s}: {_n_imgs[s]:5d} images | {n_b:6d} boxes | {n_e:4d} empty")

print(f"\nOutput directory: {OUTPUT_DIR}")


In [ ]:
# ── Fig 01: Dataset Split Overview ────────────────────────────────────────────
_splits  = ["train", "valid", "test"]
_n_img_v = [_n_imgs[s]                        for s in _splits]
_n_box_v = [(_dfs[s].class_id >= 0).sum()     for s in _splits]
_n_emp_v = [(_dfs[s].class_id <  0).sum()     for s in _splits]
_colors  = [SPLIT_PALETTE[s]                  for s in _splits]
_total_i = sum(_n_img_v)
_total_b = sum(_n_box_v)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Dataset Split Overview", fontsize=15, fontweight="bold", y=1.02)

# Panel A — images per split
ax = axes[0]
bars = ax.bar(_splits, _n_img_v, color=_colors, width=0.5, zorder=3)
for bar, n in zip(bars, _n_img_v):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f"{n:,}\n({n/_total_i*100:.1f}%)",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Images per Split")
ax.set_ylabel("Image Count")
ax.set_ylim(0, max(_n_img_v) * 1.3)

# Panel B — annotated vs hard-negatives (train only)
ax = axes[1]
_n_ann = _n_img_v[0] - _n_emp_v[0]
wedges, _, autotexts = ax.pie(
    [_n_ann, _n_emp_v[0]],
    labels=["Annotated", "Hard Negatives (empty)"],
    colors=["#2980B9", "#BDC3C7"],
    autopct="%1.1f%%", startangle=90,
    explode=(0.04, 0.04),
    textprops={"fontsize": 10},
    wedgeprops={"linewidth": 1.5, "edgecolor": "white"},
)
for at in autotexts:
    at.set_fontweight("bold")
ax.set_title("Train Split: Annotated vs Hard Negatives")

# Panel C — annotation boxes per split
ax = axes[2]
bars = ax.bar(_splits, _n_box_v, color=_colors, width=0.5, zorder=3)
for bar, n in zip(bars, _n_box_v):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 150,
            f"{n:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Annotation Boxes per Split")
ax.set_ylabel("Box Count")
ax.set_ylim(0, max(_n_box_v) * 1.2)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_01_dataset_overview.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 02: Per-Class Annotation Count (Training Set) ────────────────────────
_train_b  = _dfs["train"][_dfs["train"].class_id >= 0]
_cls_cnt  = _train_b.groupby("class_name").size().reindex(CLASS_NAMES, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 9))
y_pos = np.arange(len(CLASS_NAMES))
bars  = ax.barh(y_pos, _cls_cnt.values, color=CLASS_COLORS, height=0.68, zorder=3)

# Value labels
for bar, val in zip(bars, _cls_cnt.values):
    ax.text(bar.get_width() + 35, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", ha="left", fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels([c.replace("_", " ") for c in CLASS_NAMES], fontsize=9.5)
ax.invert_yaxis()
ax.set_xlabel("Number of Bounding Boxes")
ax.set_title("Training Set — Per-Class Annotation Count", pad=12)
ax.set_xlim(0, _cls_cnt.max() * 1.18)

# Crop separator lines (Corn=0-4, Pepper=5-14, Tomato=15-22)
for boundary in [4.5, 14.5]:
    ax.axhline(y=boundary, color="#7F8C8D", linewidth=0.9, linestyle="--", alpha=0.7)

# Low-count warning
for i, (name, cnt) in enumerate(zip(CLASS_NAMES, _cls_cnt.values)):
    if cnt < 30:
        ax.text(cnt + 35, i, "  ⚠ < 30",
                va="center", color="#E74C3C", fontsize=8.5, fontstyle="italic")

# Crop legend
patches = [
    mpatches.Patch(color=CROP_PALETTE["Corn"],   label="Corn (classes 0–4)"),
    mpatches.Patch(color=CROP_PALETTE["Pepper"], label="Pepper (classes 5–14)"),
    mpatches.Patch(color=CROP_PALETTE["Tomato"], label="Tomato (classes 15–22)"),
    mpatches.Patch(color=HEALTHY_HEX,            label="Healthy variants"),
]
ax.legend(handles=patches, loc="lower right", framealpha=0.92)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_02_class_distribution_train.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 03: Cross-Split Class Distribution (Absolute + Normalised) ────────────
_cnt = {}
for _s in ["train", "valid", "test"]:
    _b = _dfs[_s][_dfs[_s].class_id >= 0]
    _cnt[_s] = _b.groupby("class_name").size().reindex(CLASS_NAMES, fill_value=0)

_cdf  = pd.DataFrame(_cnt)
_norm = _cdf.div(_cdf.sum(axis=0), axis=1) * 100   # % within each split

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)
fig.suptitle("Class Distribution Across Dataset Splits",
             fontsize=15, fontweight="bold", y=1.01)

y_pos = np.arange(len(CLASS_NAMES))
w     = 0.27

for ax, data, xlabel, title in [
    (axes[0], _cdf,  "Annotation Box Count",       "Absolute Box Counts"),
    (axes[1], _norm, "Relative Frequency (% within split)", "Normalised Distribution"),
]:
    for i, (sp, col) in enumerate(SPLIT_PALETTE.items()):
        offset = (i - 1) * w
        ax.barh(y_pos + offset, data[sp], height=w,
                color=col, alpha=0.85, label=sp.capitalize(), zorder=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([c.replace("_", " ") for c in CLASS_NAMES], fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.legend(loc="lower right")
    for boundary in [4.5, 14.5]:
        ax.axhline(y=boundary, color="#7F8C8D", linewidth=0.8,
                   linestyle="--", alpha=0.6)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_03_cross_split_distribution.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 04: Annotation Density (Boxes per Image) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Annotation Density — Boxes per Image",
             fontsize=15, fontweight="bold")

for ax, split in zip(axes, ["train", "valid"]):
    _per_img = (_dfs[split][_dfs[split].class_id >= 0]
                .groupby("image").size())
    _mu  = _per_img.mean()
    _med = _per_img.median()
    _std = _per_img.std()

    ax.hist(_per_img.values,
            bins=range(0, int(_per_img.max()) + 2),
            color=SPLIT_PALETTE[split], alpha=0.75,
            edgecolor="white", linewidth=0.5, zorder=3,
            label=f"{split.capitalize()}  (n={len(_per_img):,} imgs)")
    ax.axvline(_mu,  color="navy",    linestyle="--", lw=2.0,
               label=f"Mean  = {_mu:.2f}")
    ax.axvline(_med, color="darkred", linestyle=":",  lw=2.0,
               label=f"Median = {_med:.0f}")

    ax.set_xlabel("Boxes per Image")
    ax.set_ylabel("Number of Images")
    ax.set_title(f"{split.capitalize()} Split  (σ = {_std:.2f})")
    ax.legend()

plt.tight_layout()
_out = OUTPUT_DIR / "fig_04_annotation_density.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 05: Bounding Box Centre Spatial Distribution ─────────────────────────
_train_box = _df_box[_df_box.split == "train"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle(
    "Bounding Box Centre Spatial Distribution (Training Set)\n"
    "Each cell = density of box centres; (0,0) = top-left, (1,1) = bottom-right",
    fontsize=12, fontweight="bold")

_panels = [("All Classes", _train_box)] + [
    (crop, _train_box[_train_box.crop == crop])
    for crop in ["Corn", "Pepper", "Tomato"]
]

for ax, (label, subset) in zip(axes, _panels):
    if len(subset) == 0:
        ax.set_visible(False)
        continue
    h = ax.hist2d(subset.cx.values, subset.cy.values,
                  bins=40, cmap="YlOrRd", density=True, cmin=1e-6)
    plt.colorbar(h[3], ax=ax, shrink=0.8, label="Density")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.invert_yaxis()
    ax.set_xlabel("cx (normalised)")
    ax.set_ylabel("cy (normalised)")
    ax.set_title(label)
    ax.set_aspect("equal")
    ax.grid(False)
    # Mark image centre
    ax.plot(0.5, 0.5, "w+", markersize=10, markeredgewidth=2)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_05_bbox_spatial_heatmap.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 06: Bounding Box Size & Aspect Ratio Analysis ────────────────────────
_tb = _df_box[_df_box.split == "train"].copy()
_tb["aspect"]  = _tb.bw / _tb.bh.clip(1e-6)
_tb["area_pct"] = _tb.area * 100          # as % of total image area

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.44, wspace=0.38)

ax_sc  = fig.add_subplot(gs[0, :2])  # scatter width vs height
ax_bp  = fig.add_subplot(gs[0, 2])   # box area by crop
ax_wh  = fig.add_subplot(gs[1, 0])   # width histogram
ax_hh  = fig.add_subplot(gs[1, 1])   # height histogram
ax_ar  = fig.add_subplot(gs[1, 2])   # aspect ratio histogram

fig.suptitle("Bounding Box Geometry Analysis — Training Set",
             fontsize=14, fontweight="bold")

# Scatter: width vs height (sampled for speed)
_samp = _tb.sample(min(6000, len(_tb)), random_state=42)
for crop, col in CROP_PALETTE.items():
    _sub = _samp[_samp.crop == crop]
    ax_sc.scatter(_sub.bw, _sub.bh, c=col, alpha=0.20, s=7, label=crop)
ax_sc.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4, label="Square (w=h)")
ax_sc.set_xlabel("Normalised Box Width (bw)")
ax_sc.set_ylabel("Normalised Box Height (bh)")
ax_sc.set_title("Box Width vs Height (6k sample)")
ax_sc.set_xlim(0, 1); ax_sc.set_ylim(0, 1)
ax_sc.legend(handles=[mpatches.Patch(color=c, label=k)
                       for k, c in CROP_PALETTE.items()],
             markerscale=2, loc="upper left")

# Box plot: area by crop
_crop_order = ["Corn", "Pepper", "Tomato"]
_bp_data    = [_tb[_tb.crop == c].area_pct.values for c in _crop_order]
_bp = ax_bp.boxplot(_bp_data, labels=_crop_order, patch_artist=True,
                    showfliers=False,
                    medianprops={"color": "black", "linewidth": 2.0})
for patch, crop in zip(_bp["boxes"], _crop_order):
    patch.set_facecolor(CROP_PALETTE[crop])
    patch.set_alpha(0.75)
ax_bp.set_ylabel("Box Area (% of image)")
ax_bp.set_title("Box Area Distribution by Crop")
ax_bp.set_xlabel("Crop")

# Histograms: width, height, aspect ratio
for ax, col, lbl, title in [
    (ax_wh, "bw",    "Normalised Width",   "Box Width Distribution"),
    (ax_hh, "bh",    "Normalised Height",  "Box Height Distribution"),
    (ax_ar, "aspect","Aspect Ratio (w/h)", "Aspect Ratio Distribution"),
]:
    for crop, color in CROP_PALETTE.items():
        _vals = _tb[_tb.crop == crop][col].dropna()
        _vals = _vals[_vals < _vals.quantile(0.99)]   # clip outliers
        ax.hist(_vals, bins=40, color=color, alpha=0.50,
                density=True, label=crop, edgecolor="none")
    ax.set_xlabel(lbl)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.savefig(OUTPUT_DIR / "fig_06_bbox_size_analysis.png")
plt.show()
print(f"Saved → {OUTPUT_DIR / 'fig_06_bbox_size_analysis.png'}")


In [ ]:
# ── Fig 07: Class Imbalance — Val/Train and Test/Train Ratios ─────────────────
_cnt2 = {}
for _s in ["train", "valid", "test"]:
    _b = _dfs[_s][_dfs[_s].class_id >= 0]
    _n = _b.groupby("class_name").size().reindex(CLASS_NAMES, fill_value=0)
    _cnt2[_s] = _n / _n.sum()   # relative frequency within split

_cdf2     = pd.DataFrame(_cnt2)
_ratio_vt = (_cdf2["valid"] / _cdf2["train"].clip(1e-9)).clip(0, 5)
_ratio_tt = (_cdf2["test"]  / _cdf2["train"].clip(1e-9)).clip(0, 5)

fig, ax = plt.subplots(figsize=(11, 8))
y_pos = np.arange(len(CLASS_NAMES))
w     = 0.36

bars_v = ax.barh(y_pos - w/2, _ratio_vt.values, height=w,
                 color=SPLIT_PALETTE["valid"], alpha=0.82,
                 label="Val / Train ratio", zorder=3)
bars_t = ax.barh(y_pos + w/2, _ratio_tt.values, height=w,
                 color=SPLIT_PALETTE["test"],  alpha=0.82,
                 label="Test / Train ratio", zorder=3)

ax.axvline(1.0, color="black", linewidth=1.2, linestyle="--",
           alpha=0.7, label="Balanced (ratio = 1.0)")
ax.axvspan(0.7, 1.3, alpha=0.06, color="green",
           label="±30 % balance zone")

ax.set_yticks(y_pos)
ax.set_yticklabels([c.replace("_", " ") for c in CLASS_NAMES], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Ratio of relative frequencies (val or test / train)")
ax.set_title("Class Imbalance — Split Frequency Ratios\n"
             "Values near 1.0 indicate well-balanced representation",
             pad=10)
ax.legend(loc="lower right")
ax.set_xlim(0, 5.2)

for boundary in [4.5, 14.5]:
    ax.axhline(y=boundary, color="#7F8C8D", linewidth=0.8,
               linestyle="--", alpha=0.6)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_07_class_imbalance.png"
plt.savefig(_out)
plt.show()
print(f"Saved → {_out}")


In [ ]:
# ── Fig 08: Training Configuration Summary Table ─────────────────────────────
_n_train_imgs = _n_imgs["train"]
_n_val_imgs   = _n_imgs["valid"]

_cfg_rows = [
    ("Model",              MODEL_SIZE,               "Fastest YOLO26 variant; ~4 MB, 3.2 GFLOPs"),
    ("Image size",         f"{IMG_SIZE}\u00d7{IMG_SIZE}",  "Matches Roboflow pre-processing resolution"),
    ("Batch size",         f"{BATCH}",               "Per-GPU; 16 images \u00d7 640\u00b2 fits 24 GB"),
    ("Epochs",             "200",                    "Cosine LR needs room to decay"),
    ("Early-stop patience","25",                     "Stop after 25 non-improving epochs"),
    ("Optimizer",          "MuSGD",                  "YOLO26 hybrid Muon + SGD"),
    ("LR (lr0 \u2192 lrf)","0.01 \u2192 0.01",       "Cosine annealing with flat final ratio"),
    ("Warmup epochs",      "5",                      "Gradual LR warmup for training stability"),
    ("Freeze layers",      "10",                     "Backbone locked; speeds up small-data training"),
    ("Close mosaic",       "10 epochs",              "Mosaic disabled in last 10 epochs (fine-tuning)"),
    ("Cache",              "RAM",                    "~4 GB; eliminates disk I/O bottleneck on Mac"),
    ("AMP",                "True (fp16)",            "\u223c1.4\u00d7 speedup on M4 Pro MPS"),
    ("Label smoothing",    "0.10",                   "Prevents overconfident predictions (OOD guard)"),
    ("Device",             "MPS (M4 Pro)",           "Apple Silicon via Metal Performance Shaders"),
    ("Train images",       f"{_n_train_imgs:,}",     "Including 338 hard-negative (empty-label) images"),
    ("Val images",         f"{_n_val_imgs:,}",       "Roboflow automatic split (\u224824 % of train+val)"),
    ("Classes",            "23",                     "Corn \u00d75, Pepper \u00d710, Tomato \u00d78"),
    ("Dataset source",     "Ghana Crop Disease v2",  "Roboflow: CC BY 4.0"),
]

fig, ax = plt.subplots(figsize=(15, 7.5))
ax.axis("off")

tbl = ax.table(
    cellText  = _cfg_rows,
    colLabels = ["Parameter", "Value", "Rationale / Notes"],
    loc       = "center",
    cellLoc   = "left",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.auto_set_column_width([0, 1, 2])

# Header row
for col in range(3):
    cell = tbl[0, col]
    cell.set_facecolor("#2C3E50")
    cell.set_text_props(color="white", fontweight="bold")
    cell.set_height(0.065)

# Alternating row colours
for row in range(1, len(_cfg_rows) + 1):
    bg = "#F4F6F7" if row % 2 == 0 else "white"
    for col in range(3):
        tbl[row, col].set_facecolor(bg)
        tbl[row, col].set_height(0.052)

ax.set_title("Training Configuration Summary",
             fontsize=14, fontweight="bold", pad=20, y=0.98)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_08_training_config.png"
plt.savefig(_out)
plt.show()
print(f"Saved \u2192 {_out}")


In [ ]:
# ── Fig 09: Learning Rate Schedule & Augmentation Profile ────────────────────
_ep_total = EPOCHS          # 200
_warmup   = 5
_lr0      = 0.01
_lrf      = 0.01
_freeze   = 10
_close_m  = 10

_ep_arr   = np.arange(1, _ep_total + 1)

def _cosine_lr(ep):
    if ep <= _warmup:
        return _lr0 * ep / _warmup
    prog = (ep - _warmup) / (_ep_total - _warmup)
    return _lr0 * (_lrf + (1 - _lrf) * 0.5 * (1 + np.cos(np.pi * prog)))

_lr_vals = np.array([_cosine_lr(e) for e in _ep_arr])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("Training Schedule & Augmentation Profile",
             fontsize=14, fontweight="bold")

# Panel A — LR curve
ax = axes[0]
ax.plot(_ep_arr, _lr_vals, color="#2980B9", linewidth=2.2, zorder=4, label="Learning Rate")
ax.fill_between(_ep_arr, 0, _lr_vals, alpha=0.12, color="#2980B9")

ax.axvspan(1,             _warmup,                alpha=0.12, color="#F39C12",
           label=f"Warmup ({_warmup} epochs)")
ax.axvspan(_warmup + 1,   _ep_total - _close_m,   alpha=0.06, color="#27AE60",
           label="Cosine Decay")
ax.axvspan(_ep_total - _close_m + 1, _ep_total,   alpha=0.12, color="#E74C3C",
           label=f"Close Mosaic (last {_close_m} ep)")
ax.axvline(_freeze, color="#8E44AD", linestyle=":", linewidth=1.8,
           label=f"Backbone Unfreeze @ ep {_freeze}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title(f"Cosine LR Schedule  (lr0={_lr0}, epochs={_ep_total})")
ax.legend(fontsize=9, loc="upper right")
ax.set_xlim(1, _ep_total)
ax.set_ylim(bottom=0)

# Panel B — Augmentation parameter profile
_aug = {
    "Mosaic":             1.00,
    "HSV Saturation":     0.70,
    "Scale":              0.50,
    "Flip LR":            0.50,
    "HSV Value":          0.40,
    "Erasing":            0.40,
    "Copy-Paste":         0.30,
    "MixUp":              0.15,
    "Label Smoothing":    0.10,
    "Translation":        0.10,
    "HSV Hue":            0.015,
    "Rotation / 90\u00b0":  10.0 / 90,
}

ax = axes[1]
_y   = np.arange(len(_aug))
_v   = list(_aug.values())
_lbl = list(_aug.keys())
_c   = ["#C0392B" if v >= 0.5 else "#2980B9" if v >= 0.1 else "#95A5A6" for v in _v]

ax.barh(_y, _v, color=_c, height=0.65, zorder=3)
ax.set_yticks(_y)
ax.set_yticklabels(_lbl, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlabel("Parameter Value (probabilities & fractions)")
ax.set_title("Augmentation Parameters")
ax.set_xlim(0, 1.18)

for i, val in enumerate(_v):
    ax.text(val + 0.015, i,
            f"{val:.3f}".rstrip("0").rstrip("."),
            va="center", fontsize=9)

# Legend for colour coding
_leg = [
    mpatches.Patch(color="#C0392B", label="Strong (\u22650.5)"),
    mpatches.Patch(color="#2980B9", label="Moderate (0.1\u2013<0.5)"),
    mpatches.Patch(color="#95A5A6", label="Mild (<0.1)"),
]
ax.legend(handles=_leg, loc="lower right", fontsize=9)

plt.tight_layout()
_out = OUTPUT_DIR / "fig_09_lr_schedule_augmentation.png"
plt.savefig(_out)
plt.show()
print(f"Saved \u2192 {_out}")


In [ ]:
# ── Publication Figure Summary ────────────────────────────────────────────────
_saved = sorted(OUTPUT_DIR.glob("fig_*.png"))
sep    = "\u2550" * 62
print(f"\n{sep}")
print(f"  Publication Figures  \u2192  {OUTPUT_DIR}")
print(sep)
for _f in _saved:
    _kb = _f.stat().st_size / 1024
    print(f"  {_f.name:<52}  {_kb:>7.1f} KB")
print(sep)
print(f"  Total : {len(_saved)} figures  |  300 DPI  |  PNG")
print(f"{sep}\n")


## 📦 Export Trained Model

Exports the best checkpoint produced above to:
- **ONNX** — universal fallback / debugging
- **ExecuTorch `.pte`** — primary Android/Kotlin target (XNNPACK backend)
- **Metadata YAML** — class names & thresholds for the Kotlin runtime

> See `export.ipynb` for the full Android integration guide and a ready-to-use `CropDiseaseDetector.kt` snippet.

In [ ]:
import shutil
import yaml
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO

# ─── Locate best checkpoint ────────────────────────────────────────────────────
best_pt = Path(results.save_dir) / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(f"best.pt not found at {best_pt}. Run training first.")

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

export_model = YOLO(str(best_pt))
print(f"Source : {best_pt}")
print(f"Dest   : {MODELS_DIR}\n")

# ─── 1. ONNX ──────────────────────────────────────────────────────────────────
print("1/2  Exporting to ONNX ...")
onnx_result = export_model.export(
    format   = "onnx",
    imgsz    = IMG_SIZE,
    dynamic  = False,      # static batch=1 for mobile inference
    simplify = True,
    opset    = 17,
    half     = False,      # FP32 for broad hardware compat
)
dst_onnx = MODELS_DIR / "crop_disease_yolo26.onnx"
shutil.copy2(str(onnx_result), dst_onnx)
print(f"   ✅  ONNX → {dst_onnx}  ({dst_onnx.stat().st_size/1_048_576:.1f} MB)")

# ─── 2. ExecuTorch (.pte) ─────────────────────────────────────────────────────
print("\n2/2  Exporting to ExecuTorch (.pte) — may take several minutes ...")
et_result = export_model.export(
    format = "executorch",
    imgsz  = IMG_SIZE,
    half   = False,
)
et_path = Path(str(et_result))
pte_src = et_path if et_path.suffix == ".pte" else next(iter(et_path.rglob("*.pte")), None)
if pte_src and pte_src.exists():
    dst_pte = MODELS_DIR / "crop_disease_yolo26.pte"
    shutil.copy2(pte_src, dst_pte)
    print(f"   ✅  ExecuTorch → {dst_pte}  ({dst_pte.stat().st_size/1_048_576:.1f} MB)")
else:
    print(f"   ⚠️  .pte not found in {et_path}. Check export output above.")
    dst_pte = None

# ─── 3. Metadata YAML ─────────────────────────────────────────────────────────
metadata = {
    "model_name"       : "crop_disease_yolo26",
    "architecture"     : "YOLO26",
    "task"             : "object_detection",
    "exported_at"      : datetime.utcnow().isoformat() + "Z",
    "input_size"       : IMG_SIZE,
    "input_channels"   : 3,
    "num_classes"      : NUM_CLASSES,
    "class_names"      : CLASS_NAMES,
    "conf_threshold"   : CONF_THRESHOLD,
    "iou_threshold"    : IOU_THRESHOLD,
    "crops_covered"    : ["Corn", "Pepper", "Tomato"],
    "notes": (
        "Trained on Ghana Crop Disease Challenge v2 (Roboflow). "
        "Hard-negative mining and label_smoothing=0.1 reduce false positives. "
        "Apply conf_threshold at inference before displaying results."
    ),
    "android_integration": {
        "runtime"         : "ExecuTorch",
        "backend"         : "XNNPACK",
        "pte_file"        : "crop_disease_yolo26.pte",
        "input_normalize" : {"mean": [0.0, 0.0, 0.0], "std": [255.0, 255.0, 255.0]},
        "input_format"    : "NCHW_RGB",
    },
}
meta_path = MODELS_DIR / "model_metadata.yaml"
with open(meta_path, "w") as f:
    yaml.dump(metadata, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
print(f"\n   ✅  Metadata → {meta_path}")

# ─── Summary ──────────────────────────────────────────────────────────────────
print("\n── Exported artefacts ──────────────────────────────────────────────────")
for file in sorted(MODELS_DIR.iterdir()):
    print(f"  {file.name:<50}  {file.stat().st_size/1_048_576:>7.1f} MB")
print("\n✅  All exports complete. Copy models/ to your Android app's assets/ folder.")
